# SFT seguido de GRPO en Kaggle (camino 3)

Parte del adaptador del SFT (`results/sft_peteroa/adaptador_sft_visible_em.zip`) y sigue con GRPO, con la misma configuración del GRPO desde el base (camino 4), salvo la duración.

Qué hace:
1. **GPU 0** entrena SFT → GRPO con la semilla 2026, en `visible` con easy y medium, durante `MINUTOS`.
2. **GPU 1**, mientras tanto, evalúa el adaptador del SFT en `eval` con el mismo protocolo que el GRPO de Kaggle (1 muestra, 4.096 tokens). Así el SFT queda medido en el mismo entorno que el base de Kaggle (53/90) y que GRPO.
3. Al terminar el entrenamiento, evalúa SFT → GRPO en `eval` con el mismo protocolo.

**Cómo usarlo:** *Settings*: **GPU T4 x2** e **Internet On**. *Add Input*: un dataset con `codigo_grpo_kaggle.zip` y `adaptador_sft_visible_em.zip` (los dos están en el repo). *Save Version > Save & Run All (Commit)*. Al terminar, en *Output* está `resultados_sft_grpo.zip` con el adaptador, las curvas y las dos evaluaciones.

Con `MINUTOS = 45` todo toma cerca de 1 h 20 min. Se probó solo la sintaxis, no en una GPU.


## 1. GPU

In [ ]:
import os
import time

INICIO = time.time()
PRUEBA_SECA = os.environ.get("PRUEBA_SECA") == "1"  # solo para probar el cuaderno sin GPU

if PRUEBA_SECA:
    N_GPU = 0
else:
    import torch

    N_GPU = torch.cuda.device_count()
    for i in range(N_GPU):
        props = torch.cuda.get_device_properties(i)
        print(f"GPU {i}: {props.name} ({props.total_memory / 2**30:.1f} GiB)")
    if N_GPU == 0:
        raise SystemExit("No hay GPU. Settings > Accelerator > GPU T4 x2.")
    if N_GPU == 1:
        print("Hay una sola GPU: se entrena una sola semilla. Con T4 x2 se entrenan dos en paralelo.")

## 2. Código

Busca en `/kaggle/input` el zip del proyecto (debe traer `scripts/train_grpo.py`), aunque Kaggle lo haya descomprimido o el navegador lo haya renombrado.

In [ ]:
import shutil
import sys
import zipfile
from pathlib import Path

ENTRADA = Path(os.environ.get("ENTRADA_KAGGLE", "/kaggle/input"))
SALIDA = Path(os.environ.get("SALIDA_KAGGLE", "/kaggle/working"))
CODIGO = Path(os.environ.get("CODIGO_KAGGLE", "/tmp/inventory_harness"))
ZIPS = CODIGO.parent / "entrada_zips"
SALIDA.mkdir(parents=True, exist_ok=True)


def archivos(raiz):
    # os.walk siguiendo enlaces: en Kaggle /kaggle/input puede tener carpetas enlazadas.
    for carpeta, _, nombres in os.walk(raiz, followlinks=True):
        for nombre in nombres:
            yield Path(carpeta) / nombre


def mostrar_entrada():
    print("Contenido de", ENTRADA, "(primeros 60 archivos):")
    for i, ruta in enumerate(sorted(archivos(ENTRADA))):
        if i == 60:
            print("  ...")
            break
        print("  ", ruta)


def es_codigo(raiz):
    return (raiz / "scripts" / "train_grpo.py").exists() and (raiz / "src" / "inventory_harness").is_dir()


shutil.rmtree(ZIPS, ignore_errors=True)
for i, ruta in enumerate(sorted(r for r in archivos(ENTRADA) if r.suffix.lower() == ".zip")):
    try:
        with zipfile.ZipFile(ruta) as archivo:
            if any(n.endswith("scripts/train_grpo.py") for n in archivo.namelist()):
                archivo.extractall(ZIPS / f"zip{i}")
                print("Descomprimido", ruta)
    except zipfile.BadZipFile:
        pass

origen = None
for raiz in (ENTRADA, ZIPS):
    for script in sorted(r for r in archivos(raiz) if r.as_posix().endswith("scripts/train_grpo.py")):
        if es_codigo(script.parent.parent):
            origen = script.parent.parent
            break
    if origen:
        break
if origen is None:
    mostrar_entrada()
    raise SystemExit("No encontre el codigo (con scripts/train_grpo.py). Revisa que el dataset tenga el zip NUEVO "
                     "y que este agregado con Add Input.")
shutil.rmtree(CODIGO, ignore_errors=True)
shutil.copytree(origen, CODIGO)
print("Codigo desde", origen)

def buscar_adaptador_sft():
    for ruta in sorted(archivos(ENTRADA)):
        if ruta.name == "adapter_config.json" and "sft" in ruta.parent.parent.name:
            return ruta.parent
    destino = CODIGO.parent / "adaptador_sft"
    shutil.rmtree(destino, ignore_errors=True)
    for ruta in sorted(r for r in archivos(ENTRADA) if r.suffix.lower() == ".zip"):
        try:
            with zipfile.ZipFile(ruta) as archivo:
                miembro = next((n for n in archivo.namelist() if n.endswith("adapter/adapter_config.json")), None)
                if miembro and "sft" in miembro:
                    archivo.extractall(destino)
                    return (destino / miembro).parent
        except zipfile.BadZipFile:
            pass
    return None


ADAPTADOR_SFT = buscar_adaptador_sft()
if ADAPTADOR_SFT is None and not PRUEBA_SECA:
    mostrar_entrada()
    raise SystemExit("No encontre el adaptador del SFT (adaptador_sft_visible_em.zip). Agregalo al dataset.")
print("Adaptador del SFT:", ADAPTADOR_SFT)


## 3. Dependencias

Versiones con las que se probaron los scripts (`requirements-train.txt`). Si no se pueden instalar, se prueba sin fijarlas, y entonces los scripts podrían necesitar ajustes menores.

In [ ]:
import subprocess

PAQUETES = ["transformers==5.24.0", "trl==1.18.2", "peft==0.22.0", "accelerate==1.16.0",
            "datasets==5.2.0", "jinja2>=3.1.6"]
if not PRUEBA_SECA:
    hecho = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PAQUETES])
    if hecho.returncode != 0:
        print("Fallo la instalacion con versiones fijas; se prueba sin fijarlas.")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers", "trl", "peft",
                        "accelerate", "datasets", "jinja2>=3.1.6"], check=True)

## 4. Configuración

Todo lo que se puede editar está en este bloque.

- **`MINUTOS`**: tiempo de entrenamiento por corrida, sin contar la carga del modelo. Las dos corridas gastan unas 3 horas de la cuota semanal de GPU (30 h) con el valor por omisión.
- **`LARGO`**: tokens máximos por respuesta, con el razonamiento incluido. Una respuesta cortada no trae `<answer>` y recibe 0. Con 2.048, la exploración cortó 37 de 180 respuestas con thinking, así que **la recompensa de partida quedará por debajo del 64 % de la línea base**, que usó 4.096. Subirlo cuesta tiempo por paso.
- **`PROMPTS_POR_PASO`**: cuántos prompts distintos entran en cada paso de optimización. Más prompts, curva menos ruidosa y pasos más lentos.
- **`NIVELES`**: con easy y medium hay señal (en easy casi todo sale bien, en hard casi todo mal); hard queda sin ver.

In [ ]:
# ----------------------------- Editar aqui -----------------------------
MODELO = "Qwen/Qwen3-1.7B"
SEMILLAS = [2026]              # una sola semilla: la GPU 1 evalua el SFT mientras tanto
MINUTOS = 45                   # el GRPO desde el base uso 150
NIVELES = ["easy", "medium"]
GRUPO = 8                      # respuestas por prompt (tamano del grupo de GRPO)
PROMPTS_POR_PASO = 2
LARGO = 2048
TASA = 2e-5                    # tasa de aprendizaje de LoRA
MAX_PASOS = 200                # tope; normalmente manda MINUTOS
POR_GPU = 2                    # respuestas por pasada en cada GPU (memoria)
EVALUAR = True                 # evalua el SFT (GPU 1) y SFT -> GRPO (al final)
LOTE_EVAL = 8
# ------------------------------------------------------------------------

if PRUEBA_SECA:  # humo en CPU con un modelo diminuto
    MODELO, MINUTOS, GRUPO, PROMPTS_POR_PASO, LARGO, MAX_PASOS = "Qwen/Qwen3-0.6B", 3, 2, 1, 32, 2

ACUM = GRUPO * PROMPTS_POR_PASO // POR_GPU
assert ACUM * POR_GPU == GRUPO * PROMPTS_POR_PASO, "GRUPO * PROMPTS_POR_PASO debe ser multiplo de POR_GPU"
USAR = SEMILLAS if PRUEBA_SECA else SEMILLAS[: max(1, N_GPU)]
RUNS = SALIDA / "runs"
LOGS = SALIDA / "logs"
TRAIN = CODIGO / "data" / "train_extra.jsonl"
print(f"Modelo {MODELO}; semillas {USAR}; {MINUTOS} min; grupo {GRUPO}; {PROMPTS_POR_PASO} prompts por paso; "
      f"largo {LARGO}; niveles {NIVELES}")

## 5. Entrenar

Genera los prompts de entrenamiento, descarga el modelo y lanza una corrida por GPU. Cada ~minuto se imprime el último paso de cada una: recompensa, desviación dentro de los grupos, fracción de grupos sin señal, largo de las respuestas y segundos por paso.

In [ ]:
import ast
import re

subprocess.run([sys.executable, "scripts/make_train_prompts.py", "--n-per-cell", "40", "--out", str(TRAIN)],
               cwd=CODIGO, check=True)
if not PRUEBA_SECA:
    subprocess.run([sys.executable, "-c",
                    f"from huggingface_hub import snapshot_download; snapshot_download({MODELO!r})"], check=True)


def resumen(linea):
    """Metricas de una linea de log del entrenador, o None si no es una."""
    try:
        datos = ast.literal_eval(linea)
    except (ValueError, SyntaxError):
        return None
    if not isinstance(datos, dict) or "reward" not in datos:
        return None

    def valor(clave):
        try:
            return float(datos[clave])
        except (KeyError, TypeError, ValueError):
            return float("nan")

    return (f"recompensa {valor('reward'):.2f} (desv {valor('reward_std'):.2f}) | "
            f"grupos sin senal {valor('frac_reward_zero_std'):.0%} | "
            f"largo {valor('completions/mean_length'):.0f} tok, cortadas {valor('completions/clipped_ratio'):.0%} | "
            f"{valor('step_time'):.0f} s/paso")


LOGS.mkdir(parents=True, exist_ok=True)
corridas = []
for indice, semilla in enumerate(USAR):
    orden = [sys.executable, "-u", "scripts/train_grpo.py", "--model", MODELO,
             "--out", RUNS / f"sft_grpo_s{semilla}", "--train-file", TRAIN, "--difficulties", *NIVELES,
             "--seed", semilla, "--num-generations", GRUPO, "--batch-size", POR_GPU, "--grad-accum", ACUM,
             "--max-completion-length", LARGO, "--learning-rate", TASA, "--max-steps", MAX_PASOS,
             "--max-minutes", MINUTOS, "--gradient-checkpointing", "--trace-every", 4, "--logging-steps", 1]
    if ADAPTADOR_SFT is not None:
        orden += ["--init-adapter", ADAPTADOR_SFT]
    entorno = dict(os.environ)
    if not PRUEBA_SECA:
        entorno["CUDA_VISIBLE_DEVICES"] = str(indice)  # una GPU por corrida
    ruta = LOGS / f"s{semilla}.log"
    manejador = open(ruta, "w")
    proceso = subprocess.Popen([str(x) for x in orden], cwd=CODIGO, env=entorno,
                               stdout=manejador, stderr=subprocess.STDOUT, text=True)
    corridas.append({"semilla": semilla, "proceso": proceso, "ruta": ruta, "visto": 0, "pasos": 0})
    print(f"Corrida con semilla {semilla} en la GPU {indice}")

EVALS = SALIDA / "eval"


def lanzar_eval(nombre, adaptador, gpu):
    carpeta = EVALS / nombre
    carpeta.mkdir(parents=True, exist_ok=True)
    orden = [sys.executable, "-u", "scripts/run_pilot.py", "--stage", "eval_final", "--models", MODELO,
             "--thinking", "on", "--conditions", "visible", "--max-new-tokens", "4096",
             "--batch-size", str(LOTE_EVAL), "--n-samples", "1", "--out", str(carpeta),
             "--adapter", str(adaptador), "--adapter-name", nombre]
    if PRUEBA_SECA:
        orden += ["--dry-run", "--limit", "9"]
    entorno = dict(os.environ, PYTHONIOENCODING="utf-8")
    if not PRUEBA_SECA:
        entorno["CUDA_VISIBLE_DEVICES"] = str(gpu)
    log = open(carpeta / "ejecucion.log", "w", encoding="utf-8")
    print(f"Evaluacion de {nombre} en la GPU {gpu}", flush=True)
    return subprocess.Popen(orden, cwd=CODIGO, env=entorno, stdout=log, stderr=subprocess.STDOUT, text=True)


EVAL_SFT = None
if EVALUAR and ADAPTADOR_SFT is not None and (N_GPU > 1 or PRUEBA_SECA):
    EVAL_SFT = lanzar_eval("sft", ADAPTADOR_SFT, 1)

limite_duro = time.time() + (MINUTOS + 45) * 60
ultimo_latido = time.time()
while any(c["proceso"].poll() is None for c in corridas):
    time.sleep(2 if PRUEBA_SECA else 60)
    for c in corridas:
        texto = c["ruta"].read_text(errors="replace")
        nuevo, c["visto"] = texto[c["visto"]:], len(texto)
        for linea in re.split(r"[\r\n]+", nuevo):
            corto = resumen(linea)
            if corto:
                c["pasos"] += 1
                print(f"[s{c['semilla']}] paso {c['pasos']}: {corto}", flush=True)
            elif "Se cumplieron" in linea:
                print(f"[s{c['semilla']}] {linea}", flush=True)
    if time.time() - ultimo_latido > 600:
        ultimo_latido = time.time()
        print(f"... {(time.time() - INICIO) / 60:.0f} min desde el inicio; pasos: "
              + ", ".join(f"s{c['semilla']}={c['pasos']}" for c in corridas), flush=True)
    if time.time() > limite_duro:
        print("Se paso del limite duro de tiempo: se detienen las corridas.")
        for c in corridas:
            c["proceso"].terminate()
        break

for c in corridas:
    c["proceso"].wait()
    codigo = c["proceso"].returncode
    print(f"Semilla {c['semilla']} termino con codigo {codigo}; {c['pasos']} pasos registrados.")
    if codigo != 0:
        print("--- ultimas lineas del log ---")
        print("\n".join(c["ruta"].read_text(errors="replace").splitlines()[-40:]))
print(f"Tiempo total: {(time.time() - INICIO) / 60:.0f} min")

## 6. Curvas

La curva de recompensa es el promedio de las respuestas de cada paso (1 = correcta). La segunda figura muestra la **fracción de grupos sin señal**: si es cercana a 1, casi todos los grupos son todo correctos o todo incorrectos y GRPO no tiene de dónde aprender. Con pocos pasos y pocos prompts por paso la recompensa sube poco o con ruido; eso no es un fallo del entorno. Lo que importa para la Entrega 2 es si hay señal y cómo se comporta.

In [ ]:
import json


def historia(carpeta):
    final = carpeta / "train_log.json"
    if final.exists():
        return json.loads(final.read_text(encoding="utf-8"))
    parcial = carpeta / "log.jsonl"
    if parcial.exists():
        return [json.loads(l) for l in parcial.read_text(encoding="utf-8").splitlines() if l.strip()]
    return []


def media(filas, clave):
    valores = []
    for fila in filas:
        try:
            valores.append(float(fila[clave]))
        except (KeyError, TypeError, ValueError):
            pass
    return sum(valores) / len(valores) if valores else float("nan")


carpetas = [d for d in sorted(RUNS.glob("sft_grpo_s*")) if any(f.get("reward") is not None for f in historia(d))]
if not carpetas:
    print("No hay pasos registrados. Mira los logs de arriba (o en logs/).")
else:
    for metrica, nombre in (("reward", "curva_recompensa.png"), ("frac_reward_zero_std", "curva_senal.png")):
        subprocess.run([sys.executable, "scripts/plot_training.py", *map(str, carpetas), "--metric", metrica,
                        "--smooth", "3", "--out", str(SALIDA / nombre)], cwd=CODIGO, check=True)
    try:
        from IPython.display import Image, display

        for nombre in ("curva_recompensa.png", "curva_senal.png"):
            if (SALIDA / nombre).exists():
                display(Image(filename=str(SALIDA / nombre)))
    except ImportError:
        pass

    print("corrida | pasos | recompensa 1er tercio -> ultimo tercio | desv media | grupos sin senal | cortadas | s/paso")
    for carpeta in carpetas:
        filas = [f for f in historia(carpeta) if "reward" in f]
        tercio = max(1, len(filas) // 3)
        print(f"{carpeta.name} | {len(filas)} | {media(filas[:tercio], 'reward'):.2f} -> "
              f"{media(filas[-tercio:], 'reward'):.2f} | {media(filas, 'reward_std'):.2f} | "
              f"{media(filas, 'frac_reward_zero_std'):.0%} | {media(filas, 'completions/clipped_ratio'):.0%} | "
              f"{media(filas, 'step_time'):.0f}")

## 7. Evaluación en eval

Espera a que termine la evaluación del SFT (GPU 1) y evalúa SFT → GRPO en la GPU 0, con el mismo protocolo del GRPO de Kaggle: las 90 instancias `visible` de `eval`, 1 muestra y 4.096 tokens. La tabla compara por dificultad; el base de Kaggle dio 24, 19 y 10 de 30 (53/90).

In [ ]:
import collections

HUELLA_EVAL = "efb77852b9c86edc5c478be0de30fa11515d288e272a6cf09f624c3bd0453e5b"
procesos = {}
if EVAL_SFT is not None:
    procesos["sft"] = EVAL_SFT
adaptadores = sorted(RUNS.glob("sft_grpo_s*/adapter"))
if EVALUAR and adaptadores:
    procesos["sft_grpo_s2026"] = lanzar_eval("sft_grpo_s2026", adaptadores[0], 0)
for nombre, proceso in procesos.items():
    print(nombre, "termino con codigo", proceso.wait(), flush=True)


def tabla(carpeta):
    manifest = json.loads((carpeta / "manifest.json").read_text(encoding="utf-8"))
    if not PRUEBA_SECA:
        assert manifest["dataset"]["fingerprint"] == HUELLA_EVAL, "La huella difiere de la linea base."
    aciertos, total = collections.Counter(), collections.Counter()
    for archivo in carpeta.glob("rollouts_*.jsonl"):
        for linea in archivo.read_text(encoding="utf-8").splitlines():
            fila = json.loads(linea)
            if fila["sample"] == 0:
                total[fila["difficulty"]] += 1
                aciertos[fila["difficulty"]] += fila["verdict"]["reward"] == 1
    return aciertos, total


print("modelo | easy | medium | hard | total")
print("base (Kaggle, 24 nov) | 24/30 | 19/30 | 10/30 | 53/90")
for nombre in procesos:
    carpeta = EVALS / nombre
    try:
        aciertos, total = tabla(carpeta)
        subprocess.run([sys.executable, "scripts/analyze_results.py", str(carpeta)], cwd=CODIGO, check=False)
        celdas = " | ".join(f"{aciertos[n]}/{total[n]}" for n in ("easy", "medium", "hard"))
        print(f"{nombre} | {celdas} | {sum(aciertos.values())}/{sum(total.values())}")
    except Exception as error:  # que el zip se arme igual
        print(nombre, "sin resultados:", error)


## 8. Resultados en un zip

In [ ]:
import datetime

ZIP = SALIDA / "resultados_sft_grpo.zip"
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as archivo:
    for ruta in sorted(RUNS.rglob("*")):
        if ruta.is_file() and "trainer" not in ruta.relative_to(RUNS).parts:
            archivo.write(ruta, "runs/" + ruta.relative_to(RUNS).as_posix())
    for ruta in sorted(LOGS.glob("*.log")):
        archivo.write(ruta, "logs/" + ruta.name)
    for ruta in sorted((SALIDA / "eval").rglob("*")):
        if ruta.is_file():
            archivo.write(ruta, "eval/" + ruta.relative_to(SALIDA / "eval").as_posix())
    for nombre in ("curva_recompensa.png", "curva_senal.png"):
        if (SALIDA / nombre).exists():
            archivo.write(SALIDA / nombre, nombre)
print(datetime.datetime.now().isoformat(timespec="seconds"))
print(f"LISTO ({ZIP.stat().st_size / 2**20:.0f} MiB): {ZIP}")
print("Descargalo desde la pestana Output de esta version, junto con curva_recompensa.png.")